# TLS provider walkthrough

This notebook opens the Ankasa TLS mesh and tree model as Xarray datasets. The large binary mesh remains memory-mapped; only small subsets are copied for plotting.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from tomo_canopy.providers import TLSProvider

## Locate the example data

Set `CRS` from the authoritative dataset metadata before integrating these coordinates with ALS, GEDI, or tomogram products.

In [ ]:
project_candidates = (Path.cwd(), Path.cwd().parent)
project_root = next(
    path for path in project_candidates if (path / "pyproject.toml").exists()
)
data_dir = project_root / "data" / "TLS" / "Ankasa"
mesh_path = data_dir / "c01_r02_subsampled_1cm_trees_mesh.ply"
trees_path = data_dir / "c01_r02_subsampled_1cm_trees.txt"
CRS = None  # Example: "EPSG:32630"; confirm this from source metadata.

assert mesh_path.exists(), f"Missing {mesh_path}"
assert trees_path.exists(), f"Missing {trees_path}"

## Open the triangular mesh

The provider exposes vertex properties and triangle connectivity without eagerly loading the full file.

In [ ]:
provider = TLSProvider()
mesh = provider.open_ply(mesh_path, crs=CRS)
mesh

In [ ]:
print(f"Geometry: {mesh.attrs['geometry_type']}")
print(f"Vertices: {mesh.sizes['vertex']:,}")
print(f"Faces: {mesh.sizes['face']:,}")
print(f"Vertex storage: {type(mesh.x.data).__name__}")
print(f"Face storage: {type(mesh.face_vertices.data).__name__}")
print("First triangle:", mesh.face_vertices.data[0])

## Plot a vertex sample

Regularly spaced indices provide a reproducible overview while limiting plotting work to 50,000 points. Coordinates are shifted locally for readable axes.

In [ ]:
sample_size = min(50_000, mesh.sizes['vertex'])
sample_index = np.linspace(
    0, mesh.sizes['vertex'] - 1, sample_size, dtype=np.int64
)
x = np.asarray(mesh.x.data[sample_index])
y = np.asarray(mesh.y.data[sample_index])
z = np.asarray(mesh.z.data[sample_index])

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(projection="3d")
points = ax.scatter(x - x.min(), y - y.min(), z, c=z, s=0.2, cmap="viridis")
ax.set(xlabel="Local x [m]", ylabel="Local y [m]", zlabel="z [m]")
fig.colorbar(points, ax=ax, label="Elevation [m]", shrink=0.7)
plt.show()

## Open the tree topology

The companion text file retains tree IDs, local segment IDs, radii, sections, and parent relationships.

In [ ]:
trees = provider.open_trees(trees_path, crs=CRS)
trees

In [ ]:
tree_count = int(trees.tree_id.max()) + 1
print(f"Trees: {tree_count:,}")
print(f"Segments: {trees.sizes['segment']:,}")
print(f"Radius range: {float(trees.radius.min()):.4f}–{float(trees.radius.max()):.4f} m")

## Plot one reconstructed tree

`parent_id` refers to the local `segment_id` within each tree. Connecting every segment to its parent reconstructs the woody topology.

In [ ]:
selected_tree = 0
indices = np.flatnonzero(trees.tree_id.data == selected_tree)
tree = trees.isel(segment=indices)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection="3d")
for segment in range(tree.sizes['segment']):
    parent = int(tree.parent_id.data[segment])
    if parent < 0:
        continue
    ax.plot(
        [tree.x.data[parent], tree.x.data[segment]],
        [tree.y.data[parent], tree.y.data[segment]],
        [tree.z.data[parent], tree.z.data[segment]],
        color="saddlebrown",
        linewidth=max(0.25, float(tree.radius.data[segment]) * 20),
    )
ax.set(xlabel="x [m]", ylabel="y [m]", zlabel="z [m]")
ax.set_title(f"TLS tree {selected_tree}")
plt.show()

## Processor hand-off

These datasets are the provider outputs intended for future TLS processors. Processing code can consume the mesh for surface sampling or voxelization and the tree model for topology- or radius-aware metrics. Keep I/O and source validation in the provider; place scientific transformations in `tomo_canopy.processors`.